# Road mirage from the ray equation

*optics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/grin_mirage)

Above a hot road the air is thinner, so its refractive index $n(y)$ dips near the ground. The ray equation of geometrical optics bends light towards higher $n$; integrating it with RK4 for a fan of viewing directions shows rays aimed at the road curving back up, so the eye sees a second, upside-down image of a distant object.

**Vector form**

$$\frac{d}{ds}\Big(n\,\frac{d\mathbf r}{ds}\Big) = \nabla n,\qquad n(y) = n_\infty - \Delta n\,e^{-y/h}$$

**Index (tensor) form**

$$\frac{dr_i}{ds} = \frac{p_i}{n},\quad \frac{dp_i}{ds} = \partial_i n,\quad p_i = n\,\frac{dr_i}{ds},\qquad p_j p_j = n^2,\quad n(y)\cos\theta = \text{const}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $r_{ri}$ | `state[:, 0, :]` | (R, 2) | position $(x, y)$ of ray $r$ |
| $p_{ri} = n\,dr_{ri}/ds$ | `state[:, 1, :]` | (R, 2) | optical direction vector: unit tangent times $n$ |
| $n(y)$ | `index(y, dn, h)` | (R,) | refractive index, lowest at the hot road surface $y = 0$ |
| $\partial_i n$ | `grad_index(r, dn, h)` | (R, 2) | only the $y$-component is non-zero: $(0,\ \tfrac{\Delta n}{h}\,e^{-y/h})$ |
| $d(r_i, p_i)/ds$ | `rhs(state, dn, h)` | (R, 2, 2) | the ray equation as a first-order system, stepped with RK4 |
| $p_j p_j = n^2$ | `einsum('rj,rj->r', p, p)` | (R,) | repeated $j$ ⇒ summed: $\lvert\mathbf p\rvert$ must stay equal to $n$ (accuracy check) |
| $n\cos\theta$ | `snell_invariant(path, dn, h)` | (S, R) | Snell's law for horizontal layers: conserved along each ray |

Rays are traced backwards from the eye (light paths are reversible): each ray answers "what does the eye see when looking in this direction?". Rays aimed slightly downward bend upward in the low-index layer and hit the object from below, so lower viewing angles show higher parts of the object — the image is inverted. Steeper rays reach the road (grey). The vertical scale is exaggerated about 100×: real index differences are $\sim 10^{-5}$ and the angles are milliradians. Computations use float64 because $n\cos\theta$ changes only in the 7th decimal place.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/grin_mirage/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dn, h = params["delta_n"] * 1e-5, params["scale_height"]
    eye, distance = params["eye_height"], params["distance"]
    angles = tensors.viewing_angles(params["max_dip"], params["n_rays"])
    state0 = tensors.initial_state(angles, eye, distance, dn, h)

    path = solve.solve(state0, dn, h, distance)
    seen = solve.height_at_object(path)

    for a, y in zip((angles * 1e3).tolist(), seen.tolist()):
        target = "road" if math.isnan(y) else f"object at {y:.2f} m"
        print(f"looking {a:+6.2f} mrad  ->  {target}")
    return plot.animate(path, angles, seen, eye, h, out_dir / "grin_mirage.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/grin_mirage/equation.py
r"""Physics → tensor translation.

In a medium whose refractive index varies smoothly, a light ray r(s) (s = arc length) obeys the
ray equation of geometrical optics,

    \frac{d}{ds}\Big(n \frac{d\mathbf r}{ds}\Big) = \nabla n

Introducing the optical direction vector p_i = n\, dr_i/ds turns it into a first-order system that
RK4 can step, for every ray r of a batch at once:

    \frac{dr_{ri}}{ds} = \frac{p_{ri}}{n(r_r)},        \frac{dp_{ri}}{ds} = \partial_i n(r_r)

Since dr/ds is a unit vector, p_{rj} p_{rj} = n^2 (repeated j ⇒ summed) must hold along each ray.

Above a hot road n depends on the height y only:  n(y) = n_\infty - \Delta n\, e^{-y/h}.
Then \partial_x n = 0, so p_x = n \cos\theta is conserved, θ being the angle to the horizontal:
Snell's law n_1 \cos\theta_1 = n_2 \cos\theta_2 for a stack of thin horizontal layers.
"""

import torch

N_INF = 1.000293  # refractive index of air far above the road


def index(y: torch.Tensor, dn: float, h: float) -> torch.Tensor:
    """n(y) = n∞ − Δn e^{−y/h}.   Any shape → same shape. (Below the road we use n(0).)"""
    return N_INF - dn * torch.exp(-y.clamp(min=0) / h)


def grad_index(r: torch.Tensor, dn: float, h: float) -> torch.Tensor:
    r"""\partial_i n = (0, Δn/h e^{−y/h}).   r: (R, 2) → (R, 2)"""
    dn_dy = dn / h * torch.exp(-r[:, 1].clamp(min=0) / h)  # (R,)
    return torch.stack([torch.zeros_like(dn_dy), dn_dy], dim=-1)


def rhs(state: torch.Tensor, dn: float, h: float) -> torch.Tensor:
    """d/ds of the state (R, 2, 2): state[:, 0] = position r_i, state[:, 1] = optical direction p_i."""
    r, p = state[:, 0], state[:, 1]
    n = index(r[:, 1], dn, h)  # (R,)
    dr_ds = p / n[:, None]  # (R, 2)
    dp_ds = grad_index(r, dn, h)  # (R, 2)
    return torch.stack([dr_ds, dp_ds], dim=1)


def direction_norm_sq(p: torch.Tensor) -> torch.Tensor:
    r"""p_{rj} p_{rj}  (should equal n²).   (..., R, 2) → (..., R)"""
    return torch.einsum("...rj,...rj->...r", p, p)


def snell_invariant(path: torch.Tensor, dn: float, h: float) -> torch.Tensor:
    """n(y) cos θ measured from the traced positions (S, R, 2) → (S − 1, R).

    θ comes from the chord between consecutive points, n from the height at the chord's midpoint.
    """
    step = path[1:] - path[:-1]  # (S − 1, R, 2)
    cos_theta = step[..., 0].abs() / step.norm(dim=-1)
    y_mid = 0.5 * (path[1:, :, 1] + path[:-1, :, 1])
    return index(y_mid, dn, h) * cos_theta


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/grin_mirage/tensors.py
"""Build the concrete tensors: the fan of viewing directions leaving the eye, as an RK4 state."""

import torch

from .equation import index

DTYPE = torch.float64  # n cos θ varies in the 7th decimal place: float32 is not enough


def viewing_angles(max_dip_mrad: float, n_rays: int) -> torch.Tensor:
    """Angles above the horizontal, from −max_dip (looking down at the road) to +2 mrad: (R,) in radians."""
    return torch.linspace(-max_dip_mrad, 2.0, n_rays, dtype=DTYPE) * 1e-3


def initial_state(angles: torch.Tensor, eye_height: float, distance: float, dn: float, h: float) -> torch.Tensor:
    """state[r] = (r_i, p_i): every ray starts at the eye (x = L, y = eye height) heading towards x = 0.

    state[r, 0, :] = (L, y_eye)                        position r_{ri}               (R, 2, 2)
    state[r, 1, :] = n(y_eye) (−cos α_r, sin α_r)      optical direction p_{ri} = n t_{ri}
    """
    r0 = torch.tensor([distance, eye_height], dtype=DTYPE).expand(len(angles), 2)
    n0 = index(torch.tensor(eye_height, dtype=DTYPE), dn, h)
    p0 = n0 * torch.stack([-torch.cos(angles), torch.sin(angles)], dim=-1)  # (R, 2)
    return torch.stack([r0, p0], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/grin_mirage/solve.py
"""Integrate the ray equation with RK4 and stop each ray at the object (x = 0) or at the road (y = 0)."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(state0: torch.Tensor, dn: float, h: float, distance: float, n_steps: int = 400) -> torch.Tensor:
    """Return the ray paths (S, R, 2), S = n_steps + 1 points along the arc length s.

    The arc length runs slightly past L so every ray reaches x = 0. Once a ray is done (x ≤ 0 or y ≤ 0)
    it is frozen at its first point beyond the boundary.
    """
    ds = 1.01 * distance / n_steps
    traj = integrate(lambda y, s: rhs(y, dn, h), state0, ds, n_steps)  # (S, R, 2, 2)
    path = traj[:, :, 0, :]  # positions only

    done = (path[..., 0] <= 0) | (path[..., 1] <= 0)  # (S, R)
    done = torch.cummax(done.int(), dim=0).values.bool()  # stays True after the first hit
    first = torch.where(done.any(0), done.int().argmax(0), len(path) - 1)  # (R,) step of the hit
    end = path[first, torch.arange(path.shape[1])]  # (R, 2)
    return torch.where(done[:, :, None], end, path)


def height_at_object(path: torch.Tensor) -> torch.Tensor:
    """Height where each ray crosses x = 0 (NaN if it hit the road first): (R,).

    Interpolate linearly between the last point before x = 0 and the first one after it.
    """
    x, y = path[..., 0], path[..., 1]  # (S, R)
    heights = torch.full((path.shape[1],), float("nan"), dtype=path.dtype)
    for r in range(path.shape[1]):
        crossed = (x[:, r] <= 0).nonzero()
        if len(crossed) == 0:
            continue
        k = crossed[0, 0]
        if y[k, r] <= 0:  # reached the road on the same step: no object hit
            continue
        w = x[k - 1, r] / (x[k - 1, r] - x[k, r])  # fraction of the step until x = 0
        heights[r] = y[k - 1, r] + w * (y[k, r] - y[k - 1, r])
    return heights


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/grin_mirage/plot.py
"""Animate rays leaving the eye and bending over the hot road (left); build up what the eye sees (right)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(path: torch.Tensor, angles: torch.Tensor, seen: torch.Tensor, eye: float, h: float, out: Path) -> Path:
    pos = path.numpy()  # (S, R, 2)
    mrad = angles.numpy() * 1e3  # (R,)
    seen = seen.numpy()  # (R,) NaN = road
    length = pos[0, 0, 0]
    y_top = max(3.5, 1.6 * eye)
    cmap = plt.get_cmap("viridis")
    colors = [cmap(y / y_top) if np.isfinite(y) else "#9ca3af" for y in seen]

    fig, (ax, ax_v) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [2.2, 1]})
    ax.set_xlim(-0.04 * length, 1.03 * length)
    ax.set_ylim(-0.15, y_top)
    ax.set_xlabel("distance x (m)")
    ax.set_ylabel("height y (m)  (exaggerated)")
    ax.axhspan(-0.15, 0, color="#4b5563")  # the road
    ax.axhspan(0, 2 * h, color="#fed7aa", alpha=0.6, zorder=0)  # the hot, low-index air layer (≈ 2 scale heights)
    ax.text(0.98 * length, 0.03, "hot air: lower n", ha="right", va="bottom", fontsize=8, color="#9a3412")
    # the object at x = 0, coloured by height with the same colour map as the rays
    ax.imshow(
        np.linspace(0, 1, 50)[:, None],
        origin="lower",
        extent=(-0.035 * length, -0.005 * length, 0, y_top),
        cmap=cmap,
        aspect="auto",
        zorder=3,
    )
    ax.plot([length], [eye], "o", color="#111827", ms=7, zorder=4)
    ax.text(length, eye + 0.12, "eye", ha="right", va="bottom", fontsize=8)
    lines = [ax.plot([], [], lw=1.1, color=c)[0] for c in colors]
    ax.set_title("rays traced back from the eye")

    ax_v.set_xlim(0, y_top)
    ax_v.set_ylim(mrad.min() - 0.5, mrad.max() + 0.5)
    ax_v.set_xlabel("object height seen (m)")
    ax_v.set_ylabel("viewing angle (mrad)")
    ax_v.set_title("what the eye sees")
    dots = ax_v.scatter([], [], s=18)

    def update(step: int):
        for r, line in enumerate(lines):
            line.set_data(pos[: step + 1, r, 0], pos[: step + 1, r, 1])
        arrived = (pos[step, :, 0] <= 0) & np.isfinite(seen)  # rays that have reached the object
        dots.set_offsets(np.column_stack([seen[arrived], mrad[arrived]]) if arrived.any() else np.empty((0, 2)))
        dots.set_color([c for c, a in zip(colors, arrived) if a])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(pos), 90), out, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'delta_n': 3.0,
    'scale_height': 0.3,
    'eye_height': 1.6,
    'distance': 400.0,
    'max_dip': 10.0,
    'n_rays': 25
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.grin_mirage")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.grin_mirage.main")
problem_dir = Path("problems/optics/grin_mirage")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
